# 00 - Setup môi trường và đọc dữ liệu (Phúc)

Đề tài 26: NER tiếng Việt trên PhoNER_COVID19. Notebook này chỉ làm 3 việc nhỏ:
1. Kiểm tra Python và GPU trên Colab.
2. Đọc 3 tập train / dev / test (mức từ và mức âm tiết), in số câu.
3. In thử 1 câu cùng nhãn để chắc là đọc đúng.

Số câu đúng theo bài báo: **5.027 / 2.000 / 3.000**.


## 1. Kiểm tra Python và GPU

Trên Colab: menu *Runtime → Change runtime type → T4 GPU* (zero-shot chạy nhanh hơn với GPU, CRF thì chỉ cần CPU).


In [1]:
import sys

print("Phiên bản Python:", sys.version.split()[0])

# torch có sẵn trên Colab; nếu chạy máy cá nhân chưa cài thì báo, không bị lỗi
try:
    import torch
    print("Phiên bản torch:", torch.__version__)
    print("Có GPU không:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Tên GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("Chưa cài torch (chỉ cần khi chạy zero-shot).")


Phiên bản Python: 3.14.3
Phiên bản torch: 2.14.1+cpu
Có GPU không: False


## 2. Đường dẫn dữ liệu và đọc 3 tập

Mặc định chạy trên máy cá nhân. Nếu chạy trên Colab thì bỏ comment 3 dòng cuối của khối đầu (phải nằm **sau** dòng mặc định để ghi đè). Dữ liệu cần nằm trong Google Drive, thư mục `NLP/PhoNER_COVID19-main/data`.

Mỗi file `.json` có mỗi dòng là 1 câu dạng `{"words": [...], "tags": [...]}`.


In [2]:
import json
import os

# Mặc định: chạy trên máy cá nhân
DATA_DIR = "PhoNER_COVID19-main/data"

# --- Nếu chạy trên Google Colab thì bỏ comment 3 dòng dưới ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/NLP/PhoNER_COVID19-main/data"


def doc_file_json(duong_dan):
    """Đọc 1 file .json của PhoNER_COVID19.
    Mỗi dòng là 1 câu dạng {"words": [...], "tags": [...]}.
    Trả về 2 list: danh sách câu (list các từ) và danh sách nhãn."""
    cac_cau = []
    cac_nhan = []
    with open(duong_dan, encoding="utf-8") as f:
        for dong in f:
            dong = dong.strip()
            if dong == "":
                continue
            mau = json.loads(dong)
            cac_cau.append(mau["words"])
            cac_nhan.append(mau["tags"])
    return cac_cau, cac_nhan


# Mức TỪ (đã tách từ) -> dùng cho CRF
train_sents, train_tags = doc_file_json(os.path.join(DATA_DIR, "word", "train_word.json"))
dev_sents,   dev_tags   = doc_file_json(os.path.join(DATA_DIR, "word", "dev_word.json"))
test_sents,  test_tags  = doc_file_json(os.path.join(DATA_DIR, "word", "test_word.json"))

# Mức ÂM TIẾT -> dùng cho zero-shot
train_syl, train_syl_tags = doc_file_json(os.path.join(DATA_DIR, "syllable", "train_syllable.json"))
dev_syl,   dev_syl_tags   = doc_file_json(os.path.join(DATA_DIR, "syllable", "dev_syllable.json"))
test_syl,  test_syl_tags  = doc_file_json(os.path.join(DATA_DIR, "syllable", "test_syllable.json"))

print("Mức từ     : train =", len(train_sents), "| dev =", len(dev_sents), "| test =", len(test_sents))
print("Mức âm tiết: train =", len(train_syl), "| dev =", len(dev_syl), "| test =", len(test_syl))


Mức từ     : train = 5027 | dev = 2000 | test = 3000
Mức âm tiết: train = 5027 | dev = 2000 | test = 3000


## 3. Xem thử 1 câu

In từng từ kèm nhãn BIO. `B-` là từ bắt đầu thực thể, `I-` là từ tiếp theo trong cùng thực thể, `O` là không thuộc thực thể nào.


In [3]:
# Câu số 3 của tập train, mức từ
cau_mau = train_sents[3]
nhan_mau = train_tags[3]
for tu, nhan in zip(cau_mau, nhan_mau):
    print(f"{tu:20s} {nhan}")

# Số từ và số nhãn phải bằng nhau
print("\nSố từ:", len(cau_mau), "| số nhãn:", len(nhan_mau))


Bà                   O
này                  O
khi                  O
trở                  O
về                   O
quá_cảnh             O
Doha                 B-LOCATION
(                    O
Qatar                B-LOCATION
)                    O
,                    O
đáp                  O
xuống                O
Tân_Sơn_Nhất         B-LOCATION
sáng                 O
2/3                  B-DATE
cùng                 O
75                   O
hành_khách           O
,                    O
trong                O
đó                   O
có                   O
55                   O
người                O
nước_ngoài           O
.                    O

Số từ: 27 | số nhãn: 27
